# Portfolio Assessment 3: Advanced ML - LSTM for Regression - Option 3b
Le Mai Chi | 105555880

This notebook is Appendix A of the report. It holds the code and stored outputs; the report holds the discussion.

In [1]:
from pathlib import Path
import os, sys, time, pickle
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # hide TensorFlow C++ log noise
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib
import tensorflow as tf
from tensorflow import keras

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

print("Python      :", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("tensorflow  :", tf.__version__)

SEED = 42
keras.utils.set_random_seed(SEED)  # python, numpy and tensorflow
tf.config.experimental.enable_op_determinism()

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

DATA = Path("../output/nasa_battery_eda_clean.csv")
OUT = Path("output/")
FIG_DIR, TAB_DIR, MODEL_DIR = OUT / "figures", OUT / "tables", OUT / "models"
for d in (FIG_DIR, TAB_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

def save_fig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

def save_table(df, name):
    df.to_csv(TAB_DIR / f"{name}.csv", index=False)
    return df

Python      : 3.12.13
scikit-learn: 1.9.1
tensorflow  : 2.21.0


## Part A: Traditional Machine Learning
### A.1 Load and analyse the dataset

In [2]:
df = pd.read_csv(DATA)
print("Shape:", df.shape, "| batteries:", df["battery_id"].nunique())
print("Columns:", list(df.columns))

assert pd.api.types.is_float_dtype(df["Capacity"])
print(f"Target Capacity: continuous, {df['Capacity'].nunique()} unique values in "
      f"[{df['Capacity'].min():.3f}, {df['Capacity'].max():.3f}] Ah -> regression")
display(df["Capacity"].describe().to_frame("Capacity (Ah)").T)

PREDICTORS = [
    "discharge_index", "ambient_temperature",
    "mean_voltage", "min_voltage", "max_voltage", "voltage_std", "voltage_drop",
    "mean_abs_current", "max_abs_current", "current_std",
    "mean_temperature", "max_temperature", "temperature_rise",
    "mean_abs_load_current", "mean_load_voltage",
]
EXCLUDED = ["capacity_class", "n_samples", "discharge_duration_s", "test_id", "uid", "filename"]
assert not set(PREDICTORS) & set(EXCLUDED + ["Capacity", "battery_id"]), "leaky column"
print("Predictors:", len(PREDICTORS), "| excluded:", EXCLUDED)

Shape: (2750, 23) | batteries: 34
Columns: ['n_samples', 'discharge_duration_s', 'mean_voltage', 'min_voltage', 'max_voltage', 'voltage_std', 'voltage_drop', 'mean_abs_current', 'max_abs_current', 'current_std', 'mean_temperature', 'max_temperature', 'temperature_rise', 'mean_abs_load_current', 'mean_load_voltage', 'battery_id', 'test_id', 'uid', 'filename', 'ambient_temperature', 'Capacity', 'discharge_index', 'capacity_class']
Target Capacity: continuous, 2750 unique values in [0.033, 2.640] Ah -> regression


,count,mean,std,min,25%,50%,75%,max
Capacity (Ah),2750.0000,1.3357,0.4611,0.0326,1.1564,1.4306,1.6744,2.6401


Predictors: 15 | excluded: ['capacity_class', 'n_samples', 'discharge_duration_s', 'test_id', 'uid', 'filename']


### A.2 Preprocess

In [3]:
X, y, groups = df[PREDICTORS], df["Capacity"], df["battery_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train = groups.iloc[train_idx]
train_bats, test_bats = sorted(g_train.unique()), sorted(groups.iloc[test_idx].unique())
assert set(train_bats).isdisjoint(test_bats), "battery leakage"

# The scaler is refitted inside every fit, so it only sees training rows
def make_pipe(model):
    return Pipeline([("scale", StandardScaler()), ("model", model)])

shapes = save_table(pd.DataFrame({
    "split": ["train", "test"],
    "batteries": [len(train_bats), len(test_bats)],
    "samples": [len(X_train), len(X_test)],
    "features": [X_train.shape[1], X_test.shape[1]],
}), "table_1_split_shapes")
display(shapes)
print("Test batteries:", test_bats)

,split,batteries,samples,features
0,train,23,1821,15
1,test,11,929,15


Test batteries: ['B0005', 'B0029', 'B0030', 'B0033', 'B0038', 'B0042', 'B0044', 'B0047', 'B0049', 'B0050', 'B0055']


### A.3 Train 2 traditional models

In [4]:
models = {
    "Random Forest": make_pipe(RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)),
    "MLP": make_pipe(MLPRegressor(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=2000, random_state=SEED)),
}
preds = {name: m.fit(X_train, y_train).predict(X_test) for name, m in models.items()}

### A.4 Evaluate and compare

### A.5 Visualise

## Part B: LSTM for Regression

### B.1 Prepare sequential data

### B.2 Build the LSTM architecture

### B.3 Train the model

### B.4 Evaluate the LSTM

### B.5 Save the models

## Part C: Final Summary and Reflection

### C.1 Master results table

### C.2 Best model
Which model performed best, why, and would you deploy it?


### C.3 Reflection questions
1. For your deep learning model: how did the architecture choices affect performance?


2. Did your deep learning model outperform the traditional ML models? Why or why not?



3. If you had more time, what would you do to improve model performance further?
